In [24]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import GridSearchCV , RandomizedSearchCV , cross_val_score, train_test_split, KFold
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
import pandas as pd
import numpy as np

In [25]:
train = pd.read_csv('dataset/train_cleaned.csv')
test = pd.read_csv('dataset/test_cleaned.csv')

In [26]:
X = train.drop('SalePrice', axis=1)
y = train['SalePrice']



In [27]:
X_test = test

In [28]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

Ridge

In [29]:
ridge_params = {
    'alpha' : [0.01 , 0.05 , 0.1,1,10,30,50,100]
}

grid_ridge = GridSearchCV(Ridge(), ridge_params, cv=kfold, scoring='neg_mean_squared_error',n_jobs=-1,verbose=1)

grid_ridge.fit(X,y)

best_ridge = grid_ridge.best_estimator_
print(f'Best alpha: {grid_ridge.best_params_}')
print(f'Best score: {grid_ridge.best_score_}')

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best alpha: {'alpha': 30}
Best score: -0.013249100949363468


Lasso

In [30]:
lasso_params = {"alpha": [0.0001, 0.0003, 0.0005, 0.0007, 0.001]}

grid_lasso = GridSearchCV(Lasso(), lasso_params, cv=kfold, scoring='neg_mean_squared_error',n_jobs=-1,verbose=1)

grid_lasso.fit(X,y)

best_lasso = grid_lasso.best_estimator_
print(f'Best alpha: {grid_lasso.best_params_}')
print(f'Best score: {grid_lasso.best_score_}')

Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best alpha: {'alpha': 0.0005}
Best score: -0.013056135119679221


XGBoost

In [31]:
xgb_model = XGBRegressor(n_estimators = 2000, learning_rate = 0.05, random_state = 42)

xgb_params = {
    'max_depth': [3, 4, 5, 6, 7],
    'min_child_weight': [1, 3, 5, 7],
    'subsample': [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9],
    'gamma': [0, 0.1, 0.2],
    'learning_rate': [0.01, 0.03, 0.05],
    'n_estimators': [1000, 2000, 3000]
}

grid_xgb = RandomizedSearchCV(xgb_model, xgb_params, cv=kfold, scoring='neg_mean_squared_error',n_jobs=-1,verbose=1)

grid_xgb.fit(X,y)

best_xgb = grid_xgb.best_estimator_
print(f'Best params: {grid_xgb.best_params_}')
print(f'Best score: {grid_xgb.best_score_}')

Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best params: {'subsample': 0.8, 'n_estimators': 3000, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 0, 'colsample_bytree': 0.7}
Best score: -0.01364917545930409


CatBoost

In [32]:
cat_model = CatBoostRegressor(verbose = 1 , random_seed = 42)
cat_params = {
    "depth": [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1],
    "iterations": [1000, 2000],
    "l2_leaf_reg": [1, 3, 5, 7]
}

grid_cat = RandomizedSearchCV(cat_model, cat_params, cv=kfold, scoring='neg_mean_squared_error',n_jobs=-1,verbose=1)

grid_cat.fit(X,y)

best_cat = grid_cat.best_estimator_
print(f'Best params: {grid_cat.best_params_}')
print(f'Best score: {grid_cat.best_score_}')

Fitting 5 folds for each of 10 candidates, totalling 50 fits
0:	learn: 0.3459113	total: 2.83ms	remaining: 2.83s
1:	learn: 0.3349144	total: 5.4ms	remaining: 2.69s
2:	learn: 0.3240871	total: 8.39ms	remaining: 2.79s
3:	learn: 0.3147219	total: 10.9ms	remaining: 2.72s
4:	learn: 0.3046122	total: 13.6ms	remaining: 2.7s
5:	learn: 0.2956266	total: 16.5ms	remaining: 2.73s
6:	learn: 0.2866494	total: 19.2ms	remaining: 2.72s
7:	learn: 0.2781838	total: 22.3ms	remaining: 2.77s
8:	learn: 0.2693929	total: 25.4ms	remaining: 2.8s
9:	learn: 0.2611489	total: 28.2ms	remaining: 2.79s
10:	learn: 0.2538420	total: 31.3ms	remaining: 2.81s
11:	learn: 0.2472319	total: 34.2ms	remaining: 2.81s
12:	learn: 0.2408762	total: 37.2ms	remaining: 2.83s
13:	learn: 0.2349682	total: 40.4ms	remaining: 2.85s
14:	learn: 0.2288143	total: 43.3ms	remaining: 2.85s
15:	learn: 0.2228661	total: 46.8ms	remaining: 2.88s
16:	learn: 0.2174771	total: 49.7ms	remaining: 2.87s
17:	learn: 0.2126113	total: 52.6ms	remaining: 2.87s
18:	learn: 0.208

LightGBM

In [33]:
lgbm_model = LGBMRegressor(random_state=42)

lgbm_params = {
    'num_leaves': [20, 31, 50],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [1000, 2000, 3000],
    'min_child_samples': [20, 30, 50],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9]
}

grid_lgbm = RandomizedSearchCV(lgbm_model, lgbm_params,
                               n_iter=20, scoring="neg_mean_squared_error",
                               cv=kfold, random_state=42, n_jobs=-1)
grid_lgbm.fit(X,y)
best_lgbm = grid_lgbm.best_estimator_
print("Best LGBM params:", grid_lgbm.best_params_)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002099 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3564
[LightGBM] [Info] Number of data points in the train set: 1402, number of used features: 197
[LightGBM] [Info] Start training from score 11.986563
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive

In [34]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score

def evaluate_model(model, X, y):
    # Get cross validation scores
    cv_scores = cross_val_score(model, X, y, cv=kfold, scoring='neg_mean_squared_error')
    rmse_scores = np.sqrt(-cv_scores)
    
    print(f'RMSE scores: {rmse_scores}')
    print(f'Mean RMSE: {rmse_scores.mean():.4f} (+/- {rmse_scores.std()*2:.4f})')

evaluate_model(best_ridge, X, y)
evaluate_model(best_lasso, X, y)
evaluate_model(best_xgb, X, y)
evaluate_model(best_cat, X, y)
evaluate_model(best_lgbm, X, y)

RMSE scores: [0.12096655 0.11232407 0.09647896 0.12345871 0.1201901 ]
Mean RMSE: 0.1147 (+/- 0.0197)


c:\Users\longo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.134e+00, tolerance: 1.468e-02
  model = cd_fast.enet_coordinate_descent(
c:\Users\longo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.720e-02, tolerance: 1.410e-02
  model = cd_fast.enet_coordinate_descent(
c:\Users\longo\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the sca

RMSE scores: [0.11689862 0.11244289 0.09629856 0.12371763 0.11996882]
Mean RMSE: 0.1139 (+/- 0.0191)
RMSE scores: [0.11615676 0.11790375 0.10162974 0.12925579 0.11998441]
Mean RMSE: 0.1170 (+/- 0.0178)
0:	learn: 0.3498321	total: 3.37ms	remaining: 3.37s
1:	learn: 0.3376225	total: 6.33ms	remaining: 3.16s
2:	learn: 0.3266701	total: 9.22ms	remaining: 3.06s
3:	learn: 0.3164006	total: 12.3ms	remaining: 3.06s
4:	learn: 0.3065712	total: 15.2ms	remaining: 3.03s
5:	learn: 0.2972204	total: 18.2ms	remaining: 3.01s
6:	learn: 0.2882965	total: 21.4ms	remaining: 3.03s
7:	learn: 0.2789459	total: 24.3ms	remaining: 3.02s
8:	learn: 0.2708771	total: 27.6ms	remaining: 3.04s
9:	learn: 0.2636375	total: 30.7ms	remaining: 3.04s
10:	learn: 0.2567303	total: 33.7ms	remaining: 3.03s
11:	learn: 0.2493787	total: 36.9ms	remaining: 3.04s
12:	learn: 0.2427100	total: 40ms	remaining: 3.03s
13:	learn: 0.2369011	total: 43.3ms	remaining: 3.05s
14:	learn: 0.2311214	total: 46.4ms	remaining: 3.05s
15:	learn: 0.2252640	total: 49

Blend

In [35]:
preds_ridge = best_ridge.predict(X_test)
preds_lasso = best_lasso.predict(X_test)
preds_xgb = best_xgb.predict(X_test)
preds_cat = best_cat.predict(X_test)
preds_lgbm = best_lgbm.predict(X_test)

final_pred = 0.2*preds_ridge + 0.3*preds_lasso + 0.3*preds_xgb + 0.1*preds_cat + 0.1*preds_lgbm

In [36]:
final_pred = np.expm1(final_pred)

In [37]:
test_Id = pd.read_csv('dataset/test.csv')['Id']

submission = pd.DataFrame({"Id": test_Id, "SalePrice": final_pred})
submission.to_csv("submission.csv", index=False)

In [ ]:
submission.describe()


,Id,SalePrice
count,1459.000000,1459.000000
mean,2190.000000,175384.955658
std,421.321334,70609.449819
min,1461.000000,49464.775231
25%,1825.500000,126174.568171
50%,2190.000000,156530.170322
75%,2554.500000,208641.578469
max,2919.000000,744692.286939
